# **Explainable AI for Student Performance Analysis**


In the age of AI-powered solutions, understanding why and how a model makes decisions is critical, especially in areas like education, healthcare, and finance. **Explainable AI (XAI)** bridges the gap between complex machine learning models and human understanding by providing tools and methods that make the decision-making process transparent and interpretable for all stakeholders, including developers, educators, and policymakers. XAI ensures that AI systems are not just accurate but also trustworthy and actionable.

Imagine you're an educator trying to identify **why some students excel while others struggle**. You want to compare a particular student to others, uncover key success factors, and determine which patterns influence academic performance. Traditional AI models provide predictions, but they rarely offer explanations. This is where **Explainable AI (XAI)** comes in.

Here you will use **IBM AI Explainability 360 (AIX360)**, a comprehensive toolkit offering state-of-the-art explainability techniques to interpret machine learning models. By applying XAI to the student performance dataset, we aim to identify representative student profiles (prototypes) that best summarize the dataset and understand the characteristics that define successful or failing students. 

Refer to the following links for detailed documentation and resources:

- [IBM AI Explainability 360 (AIX360)](https://aix360.res.ibm.com/?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-xai_heloc_practice-v1_1732308887)
- [IBM AIX360 GitHub](https://github.com/Trusted-AI/AIX360)
- [IBM AIX360 Documentation](https://aix360.readthedocs.io/_/downloads/en/latest/pdf/)


## Setup

In [1]:
!pip install --q pandas==2.2.3 | tail -n 1
!pip install --q scikit-learn==1.6.1 | tail -n 1
!pip install --q aix360==0.3.0 | tail -n 1
!pip install --q openpyxl==3.1.5 | tail -n 1
!pip install --q cvxpy==1.5.3 | tail -n 1
!pip install --q --no-deps xport==3.6.1 | tail -n 1

print('Installed!')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.
bigframes 2.26.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
   ━━━━━━

In [2]:
# Importing Reqd libraries

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from aix360.algorithms.protodash import ProtodashExplainer
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import OneHotEncoder
import matplotlib.pyplot as plt

## <a id='data-loading-and-preprocessing'></a>[Data Loading and Preprocessing](#toc)

We use the pandas library to load the student performance dataset. The data is stored in a CSV file named `student-por.csv`.

We will now load the dataset and inspect the first few rows of the data using pandas `.head()` function.

To know more about the dataset follow this [link](https://archive.ics.uci.edu/dataset/320/student+performance).


In [3]:
dataset_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/Qu8EfwuQ8OeYxaEuR8ZC9w/student-por.csv'

df = pd.read_csv(dataset_url, sep=";")
df

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
644,MS,F,19,R,GT3,T,2,3,services,other,...,5,4,2,1,2,5,4,10,11,10
645,MS,F,18,U,LE3,T,3,1,teacher,services,...,4,3,4,1,1,1,4,15,15,16
646,MS,F,18,U,GT3,T,1,1,other,other,...,1,1,1,1,1,5,6,11,12,9
647,MS,M,17,U,LE3,T,3,1,services,services,...,2,4,5,3,4,2,6,10,10,10


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 649 entries, 0 to 648
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      649 non-null    object
 1   sex         649 non-null    object
 2   age         649 non-null    int64 
 3   address     649 non-null    object
 4   famsize     649 non-null    object
 5   Pstatus     649 non-null    object
 6   Medu        649 non-null    int64 
 7   Fedu        649 non-null    int64 
 8   Mjob        649 non-null    object
 9   Fjob        649 non-null    object
 10  reason      649 non-null    object
 11  guardian    649 non-null    object
 12  traveltime  649 non-null    int64 
 13  studytime   649 non-null    int64 
 14  failures    649 non-null    int64 
 15  schoolsup   649 non-null    object
 16  famsup      649 non-null    object
 17  paid        649 non-null    object
 18  activities  649 non-null    object
 19  nursery     649 non-null    object
 20  higher    

**The data has 649 rows and 33 columns: 16 Num cols & 17 Object cols**

### <a id='dataset-description'></a>[Dataset Description](#toc)

The dataset contains data related to Portuguese language performance among secondary school students in Portugal. It includes various demographic, social, and academic attributes, providing a rich source of information for analyzing factors influencing student performance.

The `info()` function provides a concise summary of the dataset, including the number of non-null entries, data types, and memory usage. A breakdown of the dataset follows:

| Column     | Type        | Description                                                                                  | Category Types/Range                                                                                 |
|------------|-------------|----------------------------------------------------------------------------------------------|-----------------------------------------------------------------------------------------------|
| school     | Categorical | The student's school.                                                                        | 'GP' - Gabriel Pereira, 'MS' - Mousinho da Silveira                                           |
| sex        | Categorical | The student's gender.                                                                        | 'F' - Female, 'M' - Male                                                                      |
| age        | Numerical   | The student's age.                                                                           | From 15 to 22                                                                                 |
| address    | Categorical | The student's home address type.                                                             | 'U' - Urban, 'R' - Rural                                                                      |
| famsize    | Categorical | Family size.                                                                                 | 'LE3' - Less or equal to 3, 'GT3' - Greater than 3                                            |
| Pstatus    | Categorical | Parent's cohabitation status.                                                                | 'T' - Living together, 'A' - Apart                                                            |
| Medu       | Numerical   | Mother's education.                                                                          | 0 - None, 1 - Primary, 2 - 5th to 9th grade, 3 - Secondary, 4 - Higher                        |
| Fedu       | Numerical   | Father's education.                                                                          | 0 - None, 1 - Primary, 2 - 5th to 9th grade, 3 - Secondary, 4 - Higher                        |
| Mjob       | Categorical | Mother's job.                                                                                | 'teacher', 'health', 'services', 'at_home', 'other'                                           |
| Fjob       | Categorical | Father's job.                                                                                | 'teacher', 'health', 'services', 'at_home', 'other'                                           |
| reason     | Categorical | Reason for choosing this school.                                                             | 'home', 'reputation', 'course', 'other'                                                      |
| guardian   | Categorical | Student's guardian.                                                                          | 'mother', 'father', 'other'                                                                   |
| traveltime | Numerical   | Home to school travel time.                                                                  | 1 - <15 min, 2 - 15–30 min, 3 - 30–60 min, 4 - >60 min                                       |
| studytime  | Numerical   | Weekly study time.                                                                           | 1 - <2 hours, 2 - 2–5 hours, 3 - 5–10 hours, 4 - >10 hours                                   |
| failures   | Numerical   | Number of past class failures.                                                               | 0 - No failures, 1 to 3 - Actual failures, 4 - More than 3                                    |
| schoolsup  | Categorical | Extra educational support.                                                                   | 'yes', 'no'                                                                                   |
| famsup     | Categorical | Family educational support.                                                                  | 'yes', 'no'                                                                                   |
| paid       | Categorical | Extra paid classes within the course subject.                                                | 'yes', 'no'                                                                                   |
| activities | Categorical | Extra-curricular activities.                                                                 | 'yes', 'no'                                                                                   |
| nursery    | Categorical | Attended nursery school.                                                                     | 'yes', 'no'                                                                                   |
| higher     | Categorical | Wants to take higher education.                                                              | 'yes', 'no'                                                                                   |
| internet   | Categorical | Internet access at home.                                                                     | 'yes', 'no'                                                                                   |
| romantic   | Categorical | In a romantic relationship.                                                                  | 'yes', 'no'                                                                                   |
| famrel     | Numerical   | Quality of family relationships.                                                             | 1 - Very bad to 5 - Excellent                                                                 |
| freetime   | Numerical   | Free time after school.                                                                      | 1 - Very low to 5 - Very high                                                                 |
| goout      | Numerical   | Going out with friends.                                                                      | 1 - Very low to 5 - Very high                                                                 |
| Dalc       | Numerical   | Workday alcohol consumption.                                                                 | 1 - Very low to 5 - Very high                                                                 |
| Walc       | Numerical   | Weekend alcohol consumption.                                                                 | 1 - Very low to 5 - Very high                                                                 |
| health     | Numerical   | Current health status.                                                                       | 1 - Very bad to 5 - Very good                                                                 |
| absences   | Numerical   | Number of school absences.                                                                   | From 0 to 93                                                                                 |
| G1         | Numerical   | First-period grade.                                                                          | From 0 to 20                                                                                 |
| G2         | Numerical   | Second-period grade.                                                                         | From 0 to 20                                                                                 |
| G3         | Numerical   | Final grade.                                                                                 | From 0 to 20                                                                                 |

This dataset provides a comprehensive view of the socio-demographic and academic aspects influencing Portuguese language performance, ideal for educational and psychological studies. 

The target variable for our analysis will be **G3 (Final Grade)** because it represents the overall academic performance of the students and serves as the outcome we aim to predict or understand.


In [5]:
# The info tells us no missing values, so let's check duplicates

df.duplicated().sum()

np.int64(0)

In [6]:
# let's see the distribution of values in the target
df.G3.value_counts()

G3
11    104
10     97
13     82
12     72
14     63
15     49
16     36
8      35
9      35
17     29
18     15
0      15
7      10
6       3
19      2
1       1
5       1
Name: count, dtype: int64

## <a id='model-training-and-evaluation'></a>[Model Training and Evaluation](#toc)

To begin model training, we first separate the target variable `G3` from the dataset. The features are then used to predict the target variable. Next, we split the dataset into training and testing sets to evaluate the model's performance. The target variable `G3` is binarized (e.g., Pass/Fail based on a threshold) to simplify the problem into a classification task. This approach makes it easier to interpret the results and understand the model's predictions while focusing on whether a student meets a specific performance criterion.

> It is important to note that the threshold for determining "Pass" and "Fail" may vary based on specific use cases or regional standards. The current threshold is an assumption made for simplicity. You are encouraged to experiment with different thresholds to observe how the model's performance and the outputs of the Protodash Explainer change accordingly.


In [7]:
# Dropping the target column 'G3' from feature set
X = df.drop(columns=['G3'])

# Binarize G3: 1 for pass (G3 > 8), 0 for fail
y = (df['G3'] > 8).astype(int)  

In [8]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(519, 32)
(519,)
(130, 32)
(130,)


After train test split let's reset the indices to avoid the alignment issues. Meaning after splitting, subsets might carry over the original row indices from the parent DataFrame. For instance, `X_train` might have rows labeled [10, 15, 21, ...] rather than [0, 1, 2, ...]. By resetting the index, we ensure that each subset has a clean, consecutive integer index starting at zero. Then `row 0` of `X_train` corresponds exactly to `row 0` of `y_train`, etc.

We set `drop` to `True`, because it discards the old index rather than adding it back as a separate column.


In [9]:
# Reset indices to avoid alignment issues
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

### <a id='handling-categorical-features'></a>[Handling Categorical Features](#toc)

From the dataset, we observe that it contains a mix of categorical and numerical features. Among the categorical features:
- Some are **binary** (e.g., `schoolsup` - yes/no),
- Some are **multi-class** (e.g., `Mjob` - teacher, health, services, etc.),
- Some are **ordinal** (e.g., `studytime` - 1: <2 hours, 4: >10 hours),
- Others are **nominal** (e.g., `reason` - home, reputation, course, etc.).

Handling these features appropriately is crucial to ensure the model performs optimally. To achieve this, we will encode and scale these features using appropriate techniques:

1. **Encode binary features using OneHotEncoder**:
   - Converts binary categorical variables into numerical values (0 and 1).

2. **One-hot encode multi-class features**:
   - Prevents models from assuming ordinal relationships in nominal features.
   - Expands categorical features into separate binary columns for each category.

We do **not** scale ordinal or nominal features because **Random Forest** does **not** require scaled inputs. Random Forests are made up of decision trees, and decision trees split data based on the **order** of values rather than their **exact magnitudes**. In other words, if you use a strictly monotonic transformation—like multiplying all values by 2, or applying a log function to positive numbers—the **relative order** of the data points stays the same, so the trees’ split points **remain effectively the same**.

For example, consider the ordinal feature `studytime` with values from 1 to 4. If you apply a non-linear scaling, those values might change, but **1 is still less than 2, 2 is still less than 3**, and so on. Because the tree only cares about who is bigger or smaller, the splits don’t fundamentally change.

For **nominal** features (like `reason`: home, reputation, course), there is **no natural order** at all. Assigning numeric scales doesn’t help. Instead, we use **OneHotEncoder** so each category can be handled correctly by the model, without pretending there is a meaningful numeric scale.

In short, scaling these features isn’t harmful, but it’s not needed for a Random Forest. We simply **one-hot encode** the categorical variables without further numeric transformations.


In [10]:
# Define categorical features
binary_features = ['school', 'sex', 'address', 'famsize', 'Pstatus', 'schoolsup', 
                   'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']
multi_class_features = ['Mjob', 'Fjob', 'reason', 'guardian']

We are using **One-Hot Encoding** to convert **`multi-class categorical features`** into a numerical format that the model can understand. First, we **train the encoder on the training data**, so it learns which categories exist. Then, we **use the same encoder to transform the test data**, ensuring that the test set is encoded based only on the categories seen during training. 


In [11]:
# Apply One-Hot Encoding separately to train and test sets
onehot_encoder = OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore")  # handle_unknown="ignore" prevents unseen category issues

# Fit on training data and transform both train and test sets
X_train_encoded = onehot_encoder.fit_transform(X_train[multi_class_features])
X_test_encoded = onehot_encoder.transform(X_test[multi_class_features])

# Convert to DataFrame with appropriate column names
X_train_encoded_df = pd.DataFrame(X_train_encoded, columns=onehot_encoder.get_feature_names_out(multi_class_features), index=X_train.index)
X_test_encoded_df = pd.DataFrame(X_test_encoded, columns=onehot_encoder.get_feature_names_out(multi_class_features), index=X_test.index)

In [12]:
X_train_encoded2 = pd.get_dummies(X_train[multi_class_features]).astype('int')
X_test_encoded2 = pd.get_dummies(X_test[multi_class_features]).astype('int')

print(X_train_encoded2.shape)
print(X_test_encoded2.shape)

(519, 17)
(130, 17)


In [13]:
X_train_encoded2.head()

,Mjob_at_home,Mjob_health,Mjob_other,Mjob_services,Mjob_teacher,Fjob_at_home,Fjob_health,Fjob_other,Fjob_services,Fjob_teacher,reason_course,reason_home,reason_other,reason_reputation,guardian_father,guardian_mother,guardian_other
0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0
1,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,1,0
2,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0
3,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0
4,1,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0


In [14]:
X_train_encoded_df.head()

,Mjob_health,Mjob_other,Mjob_services,Mjob_teacher,Fjob_health,Fjob_other,Fjob_services,Fjob_teacher,reason_home,reason_other,reason_reputation,guardian_mother,guardian_other
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
2,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Finally, we drop the original categorical columns and add the one‐hot‐encoded columns, ensuring that both `X_train` and `X_test` match in their new feature set so the model can process them correctly.


In [15]:
g = pd.get_dummies(X_train[multi_class_features])
g.shape

(519, 17)

In [16]:
X_train_encoded_df.shape

(519, 13)

In [17]:
set(list(g)) - set(list(X_train_encoded_df))

{'Fjob_at_home', 'Mjob_at_home', 'guardian_father', 'reason_course'}

In [18]:
X_train_encoded_df.head()

,Mjob_health,Mjob_other,Mjob_services,Mjob_teacher,Fjob_health,Fjob_other,Fjob_services,Fjob_teacher,reason_home,reason_other,reason_reputation,guardian_mother,guardian_other
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
2,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [19]:
X_train.Fjob.unique()

array(['at_home', 'teacher', 'other', 'services', 'health'], dtype=object)